
# Parallelized TCR‑pMHC 5D Contact Matrix Builder

This notebook:

- Parses multiple PDB files of TCR‑pMHC complexes
- Extracts chains A–E
- Computes padded 5D binary contact matrices
- Uses **parallel CPU processing**
- Optionally uses **GPU acceleration with CuPy**
- Saves:
  - `.npz` compressed matrices
  - metadata JSON
  - optional amino‑acid sequences

---

## Biological Chain Mapping

| Chain | Role |
|---|---|
| A | MHC alpha |
| B | MHC beta |
| C | Peptide |
| D | TCR beta |
| E | TCR alpha |


In [ ]:

# ============================================================
# INSTALLS (Run once if needed)
# ============================================================

# CPU
# !pip install biopython tqdm

# GPU (optional)
# CUDA 12 example:
# !pip install cupy-cuda12x

# CUDA 11 example:
# !pip install cupy-cuda11x


In [ ]:

# ============================================================
# IMPORTS
# ============================================================

import os
import json
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
from tqdm import tqdm

# Optional GPU support
USE_GPU = False

try:
    import cupy as cp
    GPU_AVAILABLE = True
except Exception:
    GPU_AVAILABLE = False

if USE_GPU and not GPU_AVAILABLE:
    raise RuntimeError("CuPy not installed but USE_GPU=True")

print(f"GPU Available: {GPU_AVAILABLE}")
print(f"Using GPU     : {USE_GPU}")


In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================

PDB_DIR = "pdb_files"
OUTPUT_DIR = "output_5d"

DISTANCE_CUTOFF = 6.0

CHAIN_MAP = {
    "A": "MHC_alpha",
    "B": "MHC_beta",
    "C": "Peptide",
    "D": "TCR_beta",
    "E": "TCR_alpha"
}

SAVE_SEQUENCES = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

pdb_files = sorted(Path(PDB_DIR).glob("*.pdb"))

print(f"Found {len(pdb_files)} PDB files")


In [ ]:

# ============================================================
# HELPERS
# ============================================================

parser = PDBParser(QUIET=True)


def extract_chain_data(pdb_path):
    '''
    Extract residues, sequences, and CA coordinates.
    Uses model 0 only.
    '''

    structure = parser.get_structure(pdb_path.stem, pdb_path)
    model = structure[0]

    chain_data = {}

    for chain_id in CHAIN_MAP:

        if chain_id not in model:
            chain_data[chain_id] = {
                "sequence": "",
                "coords": [],
                "length": 0
            }
            continue

        chain = model[chain_id]

        sequence = []
        coords = []

        for residue in chain:

            if not is_aa(residue, standard=True):
                continue

            try:
                aa = seq1(residue.resname)
            except Exception:
                continue

            sequence.append(aa)

            if "CA" in residue:
                atom = residue["CA"]

                if atom.is_disordered():
                    atom = atom.selected_child

                coords.append(atom.coord.astype(np.float32))

            else:
                coords.append(None)

        chain_data[chain_id] = {
            "sequence": "".join(sequence),
            "coords": coords,
            "length": len(sequence)
        }

    return chain_data


In [ ]:

# ============================================================
# PASS 1 — SCAN ALL FILES FOR MAX LENGTHS
# Parallelized CPU
# ============================================================

def scan_lengths(pdb_path):

    data = extract_chain_data(pdb_path)

    return {
        "pdb_id": pdb_path.stem,
        "lengths": {
            chain: data[chain]["length"]
            for chain in CHAIN_MAP
        },
        "sequences": {
            chain: data[chain]["sequence"]
            for chain in CHAIN_MAP
        }
    }


with Pool(processes=cpu_count()) as pool:
    scan_results = list(
        tqdm(
            pool.imap(scan_lengths, pdb_files),
            total=len(pdb_files),
            desc="Scanning PDBs"
        )
    )


max_lengths = {
    chain: max(r["lengths"][chain] for r in scan_results)
    for chain in CHAIN_MAP
}

print("\nMaximum Chain Lengths")
for chain, length in max_lengths.items():
    print(f"{chain}: {length}")


In [ ]:

# ============================================================
# GPU / CPU CONTACT COMPUTATION
# ============================================================

def compute_contact_map(coords_D, coords_E, cutoff=6.0):

    len_D = len(coords_D)
    len_E = len(coords_E)

    valid_D = [i for i, c in enumerate(coords_D) if c is not None]
    valid_E = [j for j, c in enumerate(coords_E) if c is not None]

    contact_map = np.zeros((len_D, len_E), dtype=np.uint8)

    if len(valid_D) == 0 or len(valid_E) == 0:
        return contact_map

    D_array = np.array([coords_D[i] for i in valid_D], dtype=np.float32)
    E_array = np.array([coords_E[j] for j in valid_E], dtype=np.float32)

    # ---------------- GPU PATH ----------------
    if USE_GPU:

        D_gpu = cp.asarray(D_array)
        E_gpu = cp.asarray(E_array)

        diff = D_gpu[:, None, :] - E_gpu[None, :, :]
        dist = cp.linalg.norm(diff, axis=2)

        contacts = (dist < cutoff)

        contacts_cpu = cp.asnumpy(contacts)

    # ---------------- CPU PATH ----------------
    else:

        diff = D_array[:, None, :] - E_array[None, :, :]
        dist = np.linalg.norm(diff, axis=2)

        contacts_cpu = dist < cutoff

    for ii, i in enumerate(valid_D):
        for jj, j in enumerate(valid_E):
            if contacts_cpu[ii, jj]:
                contact_map[i, j] = 1

    return contact_map


In [ ]:

# ============================================================
# BUILD FULL 5D MATRIX
# ============================================================

def build_matrix_for_pdb(pdb_path):

    pdb_id = pdb_path.stem

    data = extract_chain_data(pdb_path)

    lengths = {
        c: data[c]["length"]
        for c in CHAIN_MAP
    }

    coords_D = data["D"]["coords"]
    coords_E = data["E"]["coords"]

    contact_DE = compute_contact_map(
        coords_D,
        coords_E,
        cutoff=DISTANCE_CUTOFF
    )

    shape = (
        max_lengths["A"],
        max_lengths["B"],
        max_lengths["C"],
        max_lengths["D"],
        max_lengths["E"]
    )

    matrix = np.zeros(shape, dtype=np.uint8)

    A_len = lengths["A"]
    B_len = lengths["B"]
    C_len = lengths["C"]
    D_len = lengths["D"]
    E_len = lengths["E"]

    # Efficient broadcasting
    if (
        A_len > 0 and
        B_len > 0 and
        C_len > 0 and
        D_len > 0 and
        E_len > 0
    ):

        matrix[
            :A_len,
            :B_len,
            :C_len,
            :D_len,
            :E_len
        ] = contact_DE[None, None, None, :, :]

    out_path = os.path.join(
        OUTPUT_DIR,
        f"{pdb_id}_contact_matrix.npz"
    )

    np.savez_compressed(out_path, matrix=matrix)

    return {
        "pdb_id": pdb_id,
        "lengths": lengths,
        "sequences": {
            c: data[c]["sequence"]
            for c in CHAIN_MAP
        }
    }


In [ ]:

# ============================================================
# PASS 2 — BUILD MATRICES IN PARALLEL
# ============================================================

with Pool(processes=cpu_count()) as pool:

    matrix_results = list(
        tqdm(
            pool.imap(build_matrix_for_pdb, pdb_files),
            total=len(pdb_files),
            desc="Building matrices"
        )
    )

print("\nAll matrices generated.")


In [ ]:

# ============================================================
# SAVE METADATA
# ============================================================

metadata = {
    "chain_mapping": CHAIN_MAP,
    "max_lengths": max_lengths,
    "pdbs": {}
}

for result in matrix_results:

    entry = {
        "actual_lengths": result["lengths"]
    }

    if SAVE_SEQUENCES:
        entry["sequences"] = result["sequences"]

    metadata["pdbs"][result["pdb_id"]] = entry


metadata_path = os.path.join(
    OUTPUT_DIR,
    "metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")



# Performance Notes

## CPU Parallelization

The notebook uses:

```python
multiprocessing.Pool(cpu_count())
```

This parallelizes:
- PDB parsing
- residue extraction
- contact map generation
- matrix saving

This scales very well for:
- hundreds of PDBs
- large CPU servers
- workstation clusters

---

## GPU Acceleration

If CuPy is installed:

```python
USE_GPU = True
```

Distance calculations are moved to the GPU:

```python
dist = cp.linalg.norm(diff, axis=2)
```

GPU acceleration is most useful when:
- D/E chains are large
- many complexes are processed
- using modern NVIDIA GPUs

---

## Memory Warning

A full 5D matrix can become huge.

Example:

```python
(400,400,30,250,250)
```

contains:

```python
300 billion entries
```

Even with uint8 this is enormous.

The current implementation is feasible because:
- most datasets are smaller
- matrices are compressed with NPZ
- D–E contacts are broadcast efficiently

For truly large datasets:
- use sparse tensors
- store only D–E contact maps
- generate matrices lazily
